# 7단계 실험: LangSmith Evaluation

풀 파이프라인 출력을 6개 메트릭으로 자동 평가합니다.

메트릭:
1. **grounded** — 본문 주장이 수집 자료로 뒷받침되는지 (LLM judge)
2. **citations** — `[n]` 인용 마커 비율 (programmatic)
3. **seo_title** — 제목 길이/유효성 + 주제 관련성 (mix)
4. **structure** — H1/H2/길이 (programmatic)
5. **foreigner** — 외국인 독자 이해도 (LLM judge)
6. **brand_tone** — 브랜드 톤 일관성 (LLM judge, rubric 가변)

**비용 주의**: 1회 평가 = N × 풀 파이프라인 + N×4 LLM judge. 시드 5개라면 5회 풀 파이프라인 + 20회 judge.

## 0. 환경 설정

`LANGSMITH_*` 는 LangChain import 이전에 세팅돼야 합니다.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "practice" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY",
         "LANGSMITH_TRACING", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"):
    val = os.getenv(k)
    show = val if k in ("LANGSMITH_TRACING", "LANGSMITH_PROJECT") else ("OK" if val else "MISSING")
    print(f"{k}: {show}")

## 1. 평가기 단위 검증 (programmatic)

LLM 호출 없이 가짜 출력으로 `citations` / `structure` / `seo_title` 의 일부를 확인.

In [ ]:
from llm_jacky.eval import (
    evaluate_citations, evaluate_structure, evaluate_seo_title, evaluate_grounded,
    evaluate_foreigner, evaluate_brand_tone, ALL_EVALUATORS,
)

MOCK_DRAFT = '''# 외국인을 위한 서울 길거리 음식 가이드

## 광장시장 빈대떡

광장시장에서는 빈대떡과 마약김밥이 유명합니다 [1]. 외국인들도 줄을 서서 먹는 메뉴입니다 [2].

## 명동 호떡

겨울에는 호떡이 최고입니다 [2]. 흑설탕이 입안에서 녹는 느낌입니다.

## 마무리

서울 어느 거리든 들러보세요.
'''
MOCK_OUTPUTS = {
    "draft": MOCK_DRAFT,
    "sources": [
        {"title": "Korea Tourism", "url": "https://example.com/a"},
        {"title": "Seoul Eats", "url": "https://example.com/b"},
    ],
    "seo": {
        "title": "외국인을 위한 서울 길거리 음식 가이드",
        "meta_description": "광장시장에서 명동까지, 외국인 친화 길거리 음식.",
        "tags": ["서울", "길거리음식"],
        "slug": "seoul-street-food-guide",
    },
}
MOCK_INPUTS = {"topic": "외국인을 위한 서울 길거리 음식 가이드"}

print("citations:", evaluate_citations(outputs=MOCK_OUTPUTS))
print("structure:", evaluate_structure(outputs=MOCK_OUTPUTS))

## 2. 평가기 단위 검증 (LLM judge)

OpenAI 호출이 일어남 (각 1회). 결과 점수가 합리적인지만 확인.

In [ ]:
print("grounded:    ", evaluate_grounded(outputs=MOCK_OUTPUTS))
print("seo_title:   ", evaluate_seo_title(inputs=MOCK_INPUTS, outputs=MOCK_OUTPUTS))
print("foreigner:   ", evaluate_foreigner(outputs=MOCK_OUTPUTS))
print("brand_tone:  ", evaluate_brand_tone(outputs=MOCK_OUTPUTS))

## 3. 데이터셋 업서트

LangSmith 에 `llm-jacky-blog-eval` 데이터셋을 만들고 시드 토픽들을 examples 로 등록.
재실행 시 중복 등록 안 함.

In [ ]:
from langsmith import Client
from llm_jacky.eval import upsert_dataset, DEFAULT_DATASET_NAME, SEED_TOPICS

client = Client()
ds_id = upsert_dataset(client)
print("dataset_id:", ds_id)
examples = list(client.list_examples(dataset_id=ds_id))
print(f"examples ({len(examples)}):")
for e in examples:
    print("  -", e.inputs.get("topic"))

## 4. 스모크 평가 (1~2 토픽)

비용 절약을 위해 첫 1~2개만 골라 단발 평가. 결과는 LangSmith 의 Datasets → Experiments 탭에서 표 형태로 확인.

In [ ]:
from llm_jacky.eval import run_evaluation

# 시드 첫 1개만
results = run_evaluation(topics=[SEED_TOPICS[0]], experiment_prefix="smoke")
print("experiment:", results.experiment_name)
for r in results:
    print("--- run", r.get("run").name)
    for ev in r.get("evaluation_results", {}).get("results", []):
        print(f"  {ev.key:11s} score={ev.score} comment={ev.comment[:80] if ev.comment else ''}")

## 5. 풀 평가 (5 토픽)

비용을 감수하고 시드 전체로 베이스라인 측정.

In [ ]:
# results = run_evaluation(experiment_prefix="baseline")
# print("experiment:", results.experiment_name)